# NumPy Deep Dive — Hard Exercises

**Course:** ML, Deep Learning & Computer Vision  
**Prerequisite:** Week 1 (Python + NumPy basics)  
**Estimated time:** 3–4 hours  
**Difficulty:** ★★★☆ to ★★★★  

---

This notebook drills the NumPy skills that trip up even experienced practitioners:  
broadcasting rules, advanced indexing, memory layout, stride tricks, and replacing loops with vectorised logic.  
Every exercise in deep learning — from batch normalisation to attention masks — relies on these patterns.

**Rules:**
- No Python `for`/`while` loops unless explicitly allowed.
- No Pandas, no Scikit-learn — pure NumPy.
- If you're stuck, re-read the broadcasting rules in the theory cells, then try again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

---
## Section 1 — Broadcasting

### Theory recap

Broadcasting is NumPy's mechanism for performing element-wise operations on arrays of **different shapes**.  
It avoids copying data and is the key to writing fast, loop-free code.

**The rules (memorise these):**

1. **Align dimensions from the right.** If the arrays have different numbers of dimensions, pad the shorter shape with 1s on the **left**.
2. **For each dimension pair**, the sizes must either be **equal** or one of them must be **1**.
3. A dimension of size 1 is **stretched** (broadcast) to match the other.
4. If sizes disagree and neither is 1 → **error**.

**Examples:**

| A shape | B shape | Result shape | Why |
|---------|---------|-------------|-----|
| `(3, 4)` | `(4,)` | `(3, 4)` | B padded to `(1, 4)`, row 1 broadcast to 3 |
| `(3, 4)` | `(3, 1)` | `(3, 4)` | B's col 1 broadcast to 4 |
| `(3, 1)` | `(1, 4)` | `(3, 4)` | Both broadcast |
| `(5, 3, 4)` | `(3, 4)` | `(5, 3, 4)` | B padded to `(1, 3, 4)` |
| `(3, 4)` | `(3,)` | **error** | 4 ≠ 3 and neither is 1 |

**Mental model:** Think of broadcasting as "virtual tiling". A `(3, 1)` column vector is virtually copied 4 times to fill `(3, 4)` — but NumPy never actually copies the memory.

### 1.1 Predict the shape (pen & paper first)

For each pair below, **first write your prediction** of the result shape (or "error") in the comment, **then** run the cell to check.

This is the single most important exercise in the notebook. If you can do this reliably, you can read almost any NumPy/PyTorch code.

In [ ]:
pairs = [
    ((5, 3), (3,)),         # prediction: 
    ((5, 3), (5, 1)),       # prediction: 
    ((5, 3), (1, 3)),       # prediction: 
    ((5, 3), (5,)),         # prediction: 
    ((2, 3, 4), (3, 4)),    # prediction: 
    ((2, 3, 4), (2, 1, 4)), # prediction: 
    ((2, 3, 4), (4,)),      # prediction: 
    ((2, 3, 4), (3, 1)),    # prediction: 
    ((6, 1, 4), (1, 5, 1)), # prediction: 
    ((3,), (4,)),           # prediction: 
    ((1,), (5, 3)),         # prediction: 
    ((8, 1, 6, 1), (7, 1, 5)),  # prediction: 
]

for a_shape, b_shape in pairs:
    try:
        result = np.broadcast_shapes(a_shape, b_shape)
        print(f"{str(a_shape):16s} + {str(b_shape):16s} → {result}")
    except ValueError as e:
        print(f"{str(a_shape):16s} + {str(b_shape):16s} → ERROR: {e}")

### 1.2 Row-wise and column-wise operations

Given a matrix `X` of shape `(n_samples, n_features)`:  

a) Subtract the **row mean** from each row (centre each sample)  
b) Subtract the **column mean** from each column (centre each feature)  
c) Divide each column by its **standard deviation** (standardise each feature)  
d) Divide each row by its **L2 norm** (unit-normalise each sample)  

All in one line each, using broadcasting. No loops, no `apply`.

In [ ]:
X = np.random.randn(5, 4)
print("X:\n", X)

# a) Centre each row (subtract row mean)

# b) Centre each feature (subtract column mean)

# c) Standardise each feature (subtract mean, divide by std)

# d) L2-normalise each row


### 1.3 Outer operations via broadcasting

Given two 1D arrays `a` and `b`, compute the following **without loops** and **without `np.outer`**:

a) **Outer product**: a matrix where `M[i, j] = a[i] * b[j]`  
b) **Outer addition**: `M[i, j] = a[i] + b[j]`  
c) **Outer absolute difference**: `M[i, j] = |a[i] - b[j]|`  
d) **Outer maximum**: `M[i, j] = max(a[i], b[j])`

Hint: reshape one of them to `(n, 1)` and let broadcasting do the rest.

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30])

# a) Outer product — expected shape (4, 3)

# b) Outer addition

# c) Outer absolute difference

# d) Outer maximum


### 1.4 Batch matrix operations

In deep learning, you constantly work with **batches** — a batch of images, a batch of weight matrices, a batch of feature vectors.

Given:
- `X`: shape `(batch, n, d)` — a batch of `n` vectors in `d` dimensions  
- `w`: shape `(d,)` — a single weight vector  
- `W`: shape `(d, k)` — a weight matrix  
- `b_vec`: shape `(batch, 1, d)` — a per-batch bias  

Compute the following **without loops**:

a) Weighted sum: multiply each vector by `w` element-wise, then sum over `d` → shape `(batch, n)`  
b) Linear layer: `X @ W` → shape `(batch, n, k)`  
c) Add per-batch bias: `X + b_vec` → shape `(batch, n, d)`  
d) Per-batch mean: mean of each batch's vectors → shape `(batch, d)`  
e) Cosine similarity between every pair of vectors **within each batch** → shape `(batch, n, n)`  

Hint for (e): cosine similarity = `(A @ A^T) / (norms_outer)` where norms are the L2 norms.

In [ ]:
batch, n, d, k = 3, 5, 8, 4
X = np.random.randn(batch, n, d)
w = np.random.randn(d)
W = np.random.randn(d, k)
b_vec = np.random.randn(batch, 1, d)

# a) Weighted sum → (batch, n)

# b) Linear layer → (batch, n, k)

# c) Add per-batch bias → (batch, n, d)

# d) Per-batch mean → (batch, d)

# e) Per-batch cosine similarity → (batch, n, n)


### 1.5 Attention scores (from transformers)

This is a simplified version of the **scaled dot-product attention** used in every transformer model (GPT, BERT, ViT).

Given:
- `Q`: queries, shape `(batch, seq_len, d_k)` — what each token is looking for  
- `K`: keys, shape `(batch, seq_len, d_k)` — what each token advertises  
- `V`: values, shape `(batch, seq_len, d_v)` — the actual content  

Compute:
1. Raw scores: `scores = Q @ K^T / sqrt(d_k)` → shape `(batch, seq_len, seq_len)`  
2. **Causal mask**: set `scores[i, j] = -inf` where `j > i` (can't attend to future tokens)  
3. Softmax over the last axis (per row) → `weights` shape `(batch, seq_len, seq_len)`  
4. Output: `weights @ V` → shape `(batch, seq_len, d_v)`  

Implement this **without loops**. For the mask, use broadcasting with `np.triu`.

In [ ]:
batch, seq_len, d_k, d_v = 2, 6, 8, 8
Q = np.random.randn(batch, seq_len, d_k)
K = np.random.randn(batch, seq_len, d_k)
V = np.random.randn(batch, seq_len, d_v)

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

# 1. Raw scores

# 2. Causal mask

# 3. Softmax

# 4. Output

# Verify: each row of weights should sum to 1
# Verify: weights[b, i, j] == 0 for j > i (causal)


---
## Section 2 — Advanced indexing

### Theory recap

NumPy has two kinds of advanced indexing:

**Integer array indexing** — select arbitrary elements by position:  
```python
a = np.array([10, 20, 30, 40, 50])
idx = np.array([0, 3, 4])
a[idx]  # → [10, 40, 50]
```

**Boolean indexing** — select elements where a condition is True:  
```python
a[a > 25]  # → [30, 40, 50]
```

**Fancy indexing in 2D** — index rows and columns simultaneously:  
```python
m = np.arange(12).reshape(3, 4)
m[[0, 2], [1, 3]]  # → m[0,1] and m[2,3] → [1, 11]
```

**`np.where`** — the vectorised if/else:  
```python
np.where(condition, value_if_true, value_if_false)
```

These patterns replace loops in almost every ML operation: gathering embeddings, masking padded tokens, applying class-specific weights.

### 2.1 One-hot encoding (vectorised)

Given an array of integer class labels and a number of classes, create a one-hot matrix **without loops**.

Example: `labels = [0, 2, 1, 0]`, `n_classes = 3`  
```
[[1, 0, 0],
 [0, 0, 1],
 [0, 1, 0],
 [1, 0, 0]]
```

Hint: create a zeros matrix, then use fancy indexing to set the correct positions to 1.

In [ ]:
def one_hot(labels, n_classes):
    """Vectorised one-hot encoding. No loops."""
    pass

# Test
labels = np.array([0, 2, 1, 0, 3, 2])
result = one_hot(labels, 4)
print(result)
assert result.shape == (6, 4)
assert np.all(result.sum(axis=1) == 1)  # each row has exactly one 1

### 2.2 Gather / embedding lookup

In NLP, an **embedding layer** converts token IDs into vectors by looking up rows in a matrix.  
This is just fancy indexing.

Given:
- `embedding_matrix`: shape `(vocab_size, embed_dim)` — the learned embeddings  
- `token_ids`: shape `(batch, seq_len)` — integer indices into the vocab  

Retrieve the embeddings for all tokens → result shape `(batch, seq_len, embed_dim)`.  
Do it in **one line**, no loops.

In [ ]:
vocab_size, embed_dim = 1000, 64
embedding_matrix = np.random.randn(vocab_size, embed_dim)

token_ids = np.array([[5, 102, 47, 0],
                      [88, 3, 999, 42]])

# Lookup — one line

# Verify shape


### 2.3 Cross-entropy loss (vectorised)

Implement **cross-entropy loss** for a batch of predictions, **without loops**.

Given:
- `logits`: shape `(batch, n_classes)` — raw model outputs (before softmax)  
- `targets`: shape `(batch,)` — integer class labels  

Steps:
1. Compute softmax of logits (numerically stable — subtract max per row first)  
2. Gather the predicted probability for the correct class using fancy indexing  
3. Return the **mean negative log probability**  

Formula: $\mathcal{L} = -\frac{1}{N} \sum_{i} \log(p_{i, y_i})$

In [ ]:
def cross_entropy_loss(logits, targets):
    """Vectorised cross-entropy. No loops."""
    pass

# Test
logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.3],
                   [1.0, 1.0, 3.0]])
targets = np.array([0, 1, 2])  # correct classes

loss = cross_entropy_loss(logits, targets)
print(f"Loss: {loss:.4f}")
# Expected: ~0.3068 (the model is fairly confident on the right classes)

### 2.4 Padding and masking

In NLP, sequences in a batch have different lengths and must be **padded** to the same length. A **mask** marks which positions are real vs padding.

Given a list of 1D arrays of different lengths:

a) Pad them into a single 2D array of shape `(batch, max_len)` using a pad value of 0  
b) Create a boolean mask of shape `(batch, max_len)` where `True` = real token, `False` = padding  
c) Compute the **mean of real values only** for each row, using the mask with broadcasting  

One loop is allowed for (a) — the padding step. Everything else must be vectorised.

In [ ]:
sequences = [
    np.array([4, 7, 2]),
    np.array([1, 9, 3, 5, 8]),
    np.array([6, 2]),
    np.array([3, 1, 7, 4]),
]

# a) Pad to (4, max_len)

# b) Boolean mask (True = real)

# c) Mean of real values per row (using mask + broadcasting)


### 2.5 Top-k selection per row

Given a 2D array of shape `(n, m)`, find the **indices and values of the top-k largest elements** in each row, without loops.

Return:
- `top_indices`: shape `(n, k)` — column indices of the top-k per row  
- `top_values`: shape `(n, k)` — the corresponding values, sorted descending  

Hint: `np.argpartition` is faster than sorting for top-k, but `np.argsort` is fine here.

In [ ]:
def topk_per_row(arr, k):
    """Return (top_indices, top_values) for each row. No loops."""
    pass

# Test
np.random.seed(42)
data = np.random.randn(4, 8)
print("Data:\n", data.round(2))

idx, vals = topk_per_row(data, 3)
print("\nTop-3 indices:\n", idx)
print("Top-3 values:\n", vals.round(2))

# Verify: values should be sorted descending in each row
assert np.all(np.diff(vals, axis=1) <= 0), "Values should be descending"

---
## Section 3 — Vectorisation challenges

Replace loops with broadcasting and vectorised operations. These simulate real operations you'll encounter in ML codebases.

### 3.1 Pairwise squared distances (no loops)

Given `A` shape `(n, d)` and `B` shape `(m, d)`, compute the matrix `D` of shape `(n, m)` where `D[i, j] = ||A[i] - B[j]||²`.

Do it **three different ways** (all without loops), then benchmark them:

1. **Expand and subtract**: reshape to `(n, 1, d)` and `(1, m, d)`, subtract, square, sum  
2. **Algebraic trick**: `||a - b||² = ||a||² + ||b||² - 2a·b` — uses matrix multiply, faster for large arrays  
3. **`np.einsum`**: learn the Einstein summation notation

Verify all three give the same answer.

In [ ]:
A = np.random.randn(100, 16)
B = np.random.randn(80, 16)

# Method 1: Expand and subtract

# Method 2: Algebraic trick

# Method 3: einsum

# Verify all three match

# Benchmark (use %%timeit or time.time)


### 3.2 2D convolution (no loops)

Implement a basic **2D convolution** on a single-channel image without loops. This is the core operation of CNNs.

Given:
- `image`: shape `(H, W)` — a grayscale image  
- `kernel`: shape `(kH, kW)` — a small filter (e.g. 3×3)  

Produce an output of shape `(H - kH + 1, W - kW + 1)` where each output pixel is the sum of element-wise products of the kernel with the corresponding image patch.

**Strategy:** Use `np.lib.stride_tricks.sliding_window_view` to extract all patches as a 4D array, then multiply with the kernel and sum.

Apply your function with:
- A horizontal edge-detection kernel: `[[-1, -1, -1], [0, 0, 0], [1, 1, 1]]`  
- A vertical edge-detection kernel  
- A Gaussian blur 3×3 kernel  

Visualise the results.

In [ ]:
def conv2d(image, kernel):
    """2D convolution without loops. Uses stride tricks."""
    pass

# Create a test image (checkerboard)
img = np.zeros((64, 64))
img[::8, :] = 1
img[:, ::8] = 1

# Kernels
horiz_edge = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float)
vert_edge = horiz_edge.T
gaussian = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=float) / 16

# Apply and visualise


### 3.3 Batch normalisation (vectorised)

Implement **batch normalisation** — a critical operation in modern neural networks.

Given a batch of feature maps `X` of shape `(batch, channels, height, width)`:  

1. Compute the **mean** and **variance** per channel (across batch, height, width)  
2. Normalise: $\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$  
3. Scale and shift: $y = \gamma \hat{x} + \beta$ where `gamma` and `beta` are per-channel parameters

All using broadcasting. The shapes of `mean`, `var`, `gamma`, `beta` should be `(1, channels, 1, 1)` to broadcast correctly.

In [ ]:
def batch_norm(X, gamma, beta, eps=1e-5):
    """Batch normalisation for 4D feature maps. No loops."""
    pass

# Test
batch, channels, H, W = 8, 3, 16, 16
X = np.random.randn(batch, channels, H, W) * 5 + 10  # large mean and std
gamma = np.ones((1, channels, 1, 1))
beta = np.zeros((1, channels, 1, 1))

Y = batch_norm(X, gamma, beta)
print("Input  — per-channel mean:", X.mean(axis=(0, 2, 3)).round(2))
print("Input  — per-channel std: ", X.std(axis=(0, 2, 3)).round(2))
print("Output — per-channel mean:", Y.mean(axis=(0, 2, 3)).round(4))
print("Output — per-channel std: ", Y.std(axis=(0, 2, 3)).round(4))
# Output means should be ≈ 0, stds ≈ 1

### 3.4 Non-max suppression (NMS)

Implement **non-max suppression** — the algorithm used in object detection to remove overlapping bounding boxes.

Given:
- `boxes`: shape `(N, 4)` — bounding boxes as `[x1, y1, x2, y2]`  
- `scores`: shape `(N,)` — confidence scores  
- `iou_threshold`: float — boxes with IoU above this are suppressed  

Steps:
1. Compute the **IoU (Intersection over Union)** matrix between all pairs of boxes — **vectorised** using broadcasting  
2. Greedily select boxes: pick the highest-scoring box, suppress all boxes with IoU > threshold, repeat  

The IoU computation must be fully vectorised (no loops). The greedy selection may use a loop (it's inherently sequential).

**IoU formula:**  
- Intersection area = `max(0, min(x2_a, x2_b) - max(x1_a, x1_b)) * max(0, min(y2_a, y2_b) - max(y1_a, y1_b))`  
- Union area = area_a + area_b - intersection  
- IoU = intersection / union

In [ ]:
def compute_iou_matrix(boxes):
    """Vectorised IoU matrix. boxes: (N, 4) as [x1, y1, x2, y2]. Returns: (N, N)."""
    pass

def nms(boxes, scores, iou_threshold=0.5):
    """Non-max suppression. Returns indices of kept boxes."""
    pass

# Test
boxes = np.array([
    [100, 100, 210, 210],  # box 0
    [105, 108, 215, 215],  # box 1 — heavily overlaps 0
    [300, 300, 400, 400],  # box 2 — separate
    [102, 102, 212, 212],  # box 3 — heavily overlaps 0
    [305, 305, 405, 405],  # box 4 — overlaps 2
], dtype=float)
scores = np.array([0.9, 0.75, 0.95, 0.8, 0.6])

kept = nms(boxes, scores, iou_threshold=0.5)
print("Kept box indices:", kept)
# Expected: [2, 0] — box 2 (highest score, separate) and box 0 (next highest after suppressing 1 and 3)

### 3.5 Image mosaic / grid

Given a batch of images of shape `(N, H, W, C)`, arrange them into a rectangular grid for display, using **only NumPy reshaping and transpose** — no loops, no concatenation.

For example, 16 images of shape `(32, 32, 3)` should become a `(4*32, 4*32, 3)` = `(128, 128, 3)` grid.

Steps:
1. Decide grid dimensions: `nrows × ncols = N` (pick the nearest square)  
2. Reshape `(N, H, W, C)` → `(nrows, ncols, H, W, C)`  
3. Transpose to `(nrows, H, ncols, W, C)`  
4. Reshape to `(nrows*H, ncols*W, C)`  

Test with random images and display the result.

In [ ]:
def make_grid(images):
    """Arrange (N, H, W, C) images into a grid. No loops."""
    pass

# Test: 16 random colour images, 32×32
np.random.seed(42)
images = np.random.randint(0, 256, (16, 32, 32, 3), dtype=np.uint8)

grid = make_grid(images)
print("Grid shape:", grid.shape)  # should be (128, 128, 3)

# Display


---
## Section 4 — Memory & performance

### Theory recap

Understanding how NumPy stores data in memory makes you a better debugger and optimizer.

**Key concepts:**
- **Contiguous memory**: arrays are stored as a flat buffer. The `strides` tuple tells NumPy how many bytes to skip to move one step along each axis.
- **Views vs copies**: slicing creates a **view** (shares memory). Fancy indexing creates a **copy** (new memory).
- **Row-major (C order)** vs **column-major (F order)**: C order = last axis varies fastest. NumPy defaults to C order.
- **`np.lib.stride_tricks.as_strided`**: create exotic views into memory without copying — the basis of sliding window operations.

### 4.1 Views vs copies — predict the output

For each code block, predict whether `b` is a **view** or a **copy** of `a`. Then check with `np.shares_memory(a, b)`.  
Write your prediction as a comment before running.

In [ ]:
a = np.arange(20).reshape(4, 5)

# 1. Basic slice
b = a[1:3, 2:4]
# prediction: view or copy?
print("1.", np.shares_memory(a, b))

# 2. Fancy indexing
b = a[[0, 2], :]
# prediction:
print("2.", np.shares_memory(a, b))

# 3. Boolean indexing
b = a[a > 10]
# prediction:
print("3.", np.shares_memory(a, b))

# 4. Transpose
b = a.T
# prediction:
print("4.", np.shares_memory(a, b))

# 5. Reshape
b = a.reshape(2, 10)
# prediction:
print("5.", np.shares_memory(a, b))

# 6. Flatten
b = a.flatten()
# prediction:
print("6.", np.shares_memory(a, b))

# 7. Ravel
b = a.ravel()
# prediction:
print("7.", np.shares_memory(a, b))

### 4.2 Sliding window via stride tricks

Use `np.lib.stride_tricks.sliding_window_view` (NumPy ≥ 1.20) to extract **all 3×3 patches** from a 2D array without copying memory.

Given a `(8, 8)` array:
a) Extract all 3×3 windows → shape should be `(6, 6, 3, 3)`  
b) Compute the mean of each window → `(6, 6)` result  
c) Find the window with the highest sum — report its top-left position  
d) Verify that the sliding window is a **view** (no memory copy)

In [ ]:
arr = np.random.randint(0, 100, (8, 8))
print("Array:\n", arr)

# a) Extract 3×3 windows

# b) Mean of each window

# c) Window with highest sum

# d) Verify view (no copy)


### 4.3 Memory-efficient operations

You have a large array `X` of shape `(10000, 1000)` (10M elements). Compare memory usage and speed of these approaches for computing `(X - X.mean(axis=0)) / X.std(axis=0)`:

a) **Naive**: create intermediate arrays for mean, std, centred, and final  
b) **In-place**: use `-=` and `/=` operators to minimise extra allocations  
c) **`np.subtract` and `np.divide` with `out=`**: write results into pre-allocated arrays  

Use `X.nbytes` to report memory, and `time.time` to benchmark. The in-place version should use roughly half the extra memory.

In [ ]:
import time

X = np.random.randn(10000, 1000)
print(f"X size: {X.nbytes / 1e6:.1f} MB")

# a) Naive

# b) In-place

# c) With out= parameter

# Compare results and timings


---
## Section 5 — Capstone: K-means clustering from scratch

Implement the **full K-means algorithm** using only NumPy, with **zero loops over data points** (loops over iterations are fine).

Steps per iteration:
1. **Assign** each point to the nearest centroid → uses pairwise distances (broadcasting)  
2. **Update** each centroid to the mean of its assigned points → uses fancy indexing and broadcasting  

Requirements:
- Distance computation must be fully vectorised  
- Centroid update must be fully vectorised (hint: `np.bincount` with weights, or boolean masking)  
- Track the **inertia** (total within-cluster sum of squared distances) at each step  
- Run for 50 iterations or until convergence (centroids don't change)  
- Visualise: show the clustering result and the inertia curve

Test on a synthetic 2D dataset with 4 clusters.

In [ ]:
def kmeans(X, k, max_iter=50, tol=1e-6):
    """K-means clustering. Returns (labels, centroids, inertia_history)."""
    pass

# Generate test data: 4 clusters
np.random.seed(42)
centres = np.array([[2, 2], [-2, 2], [2, -2], [-2, -2]])
X = np.vstack([c + np.random.randn(100, 2) * 0.6 for c in centres])

# Run K-means

# Plot: clustering result (left) and inertia curve (right)


---
## Submission checklist

- [ ] All cells run top-to-bottom without errors
- [ ] **No `for`/`while` loops** over data (except where explicitly allowed)
- [ ] All assertions pass
- [ ] Shapes are printed and verified for broadcasting exercises
- [ ] All plots have titles and labels

**Good luck!**